# 02 — Classical baselines: from sanity floor to a strong tabular model

A GNN is useful only if it improves on credible non-graph alternatives. This studybook builds three progressively stronger references on the exact cohort defined in `01_temporal_split_and_metrics.ipynb`.

By the end, you should understand:

1. what a no-skill base-rate model establishes;
2. how logistic regression turns a weighted linear combination into probability-like scores;
3. how gradient-boosted trees learn nonlinear rules and interactions;
4. why model selection needs its own temporal validation period;
5. what LightGBM must and must not receive for a fair GNN comparison;
6. how to interpret overall, seen-company, and cold-start results.

## 1. Why three baselines?

A single weak comparison can make any sophisticated model look impressive. These references answer different questions:

| Model | Inputs | Question |
|---|---|---|
| Base rate | none beyond training prevalence | Does the metric pipeline beat no ranking at all? |
| Logistic regression | instrument features only | Is a linear combination of invoice attributes enough? |
| LightGBM | instrument + seller/buyer company-history features | Can a strong nonlinear tabular learner already capture the available signal? |

The GNN will later receive the same instrument and company tensors as LightGBM. The difference is representational: LightGBM sees the two endpoint history vectors concatenated into a row, while the GNN performs shared message passing over typed edges.

## 2. Base-rate model

The trivial model assigns every instrument the mature training prevalence, about 1.68%. Since all scores tie, it has no ranking information: ROC AUC is 0.5 and average precision equals the evaluated cohort prevalence. Its top-k membership follows stable row order and is not operationally meaningful; top-k is useful only once scores differentiate cases.

## 3. Logistic regression

Logistic regression learns a linear score $z = b + \mathbf{w}^T\mathbf{x}$ and passes it through the sigmoid

$$P(y=1 \mid \mathbf{x}) = \sigma(z) = \frac{1}{1+e^{-z}}.$$

The decision boundary is linear in feature space, although the sigmoid is nonlinear. Here it receives only the 12 cutoff-fitted instrument features. `class_weight=balanced` gives rare impairments more influence during fitting; ranking metrics remain evaluated on the untouched natural test distribution.

## 4. Gradient boosting and LightGBM

A decision tree partitions feature space with rules such as `buyer history count < c`. One tree is unstable and limited. **Gradient boosting** builds trees sequentially: each new tree focuses on errors left by the current ensemble. The final score is an additive model

$$F_M(\mathbf{x}) = F_0(\mathbf{x}) + \eta \sum_{m=1}^{M} f_m(\mathbf{x}),$$

where $f_m$ is a tree and $\eta$ is the learning rate. LightGBM is an efficient implementation with strong performance on heterogeneous tabular data and nonlinear interactions.

Its 32 inputs are the 12 instrument features plus ten pre-cutoff history features for the seller endpoint and ten for the buyer endpoint. These histories contain origination attributes and counts—not eventual outcomes or the original thesis's hand-engineered bond-graph features.

## 5. Early stopping without touching the test period

The number of boosting trees is a model choice. Choosing it on test PR-AUC would tune the model to the answers it is supposed to predict. Instead, the latest 20% of mature pre-cutoff instruments forms a temporal validation block:

```mermaid
flowchart LR
    F[earliest 80% mature train<br/>fit candidate trees] --> V[latest 20% mature train<br/>choose tree count]
    V --> R[refit chosen tree count<br/>on all mature train]
    R --> T[score post-cutoff test<br/>once]
```

Early stopping monitors validation average precision. After selecting $M$, a fresh model with exactly $M$ trees is fitted on all 42,321 mature training instruments. The implementation is deterministic, single-threaded, and seeded.

## 6. Run the fixed baseline protocol

The reusable code lives in `src/graph_ml/baselines/tabular.py`. Outputs below are aggregate metrics and feature gains only.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from graph_ml.baselines import (
    BaselineConfig,
    assemble_tabular_features,
    evaluate_baseline_run,
    fit_tabular_baselines,
)
from graph_ml.data import GraphBuildConfig, build_trade_finance_graph
from graph_ml.evaluation import (
    TemporalSplitConfig,
    build_temporal_evaluation_split,
)

repo_root = next(
    parent
    for parent in (Path.cwd(), *Path.cwd().parents)
    if (parent / "pyproject.toml").exists()
)
data_path = repo_root / "data/02_instrumentsdf_2.parquet"
if not data_path.exists():
    raise FileNotFoundError(
        "The real local Parquet data is required; see wiki/this-project/data-availability.md"
    )

instruments = pd.read_parquet(data_path)
graph_result = build_trade_finance_graph(
    instruments, GraphBuildConfig(cutoff="2018-04-30")
)
split = build_temporal_evaluation_split(
    instruments,
    graph_result,
    TemporalSplitConfig(analysis_date="2018-12-18"),
)
features = assemble_tabular_features(graph_result)
run = fit_tabular_baselines(graph_result, split, BaselineConfig(seed=42))
{
    "instrument_only_shape": features.instrument_only.shape,
    "instrument_company_shape": features.instrument_company.shape,
    "train_prevalence": run.train_prevalence,
    "validation_start_date": run.validation_start_date.date().isoformat(),
    "lightgbm_best_iteration": run.lightgbm_best_iteration,
    "seed": run.seed,
}

{'instrument_only_shape': (59820, 12),
 'instrument_company_shape': (59820, 32),
 'train_prevalence': 0.016776541197041656,
 'validation_start_date': '2017-12-05',
 'lightgbm_best_iteration': 202,
 'seed': 42}

Early stopping selected 202 trees using validation instruments from 2017-12-05 onward. This is fit metadata, not a result chosen from test performance.

In [2]:
report = evaluate_baseline_run(
    run,
    graph_result.graph["instrument"].y.numpy(),
    split,
    review_fraction=0.05,
)
display_columns = [
    "model",
    "cohort",
    "sample_count",
    "prevalence",
    "pr_auc",
    "roc_auc",
    "top_k",
    "precision_at_k",
    "recall_at_k",
]
report[display_columns].style.format(
    {
        "prevalence": "{:.2%}",
        "pr_auc": "{:.3f}",
        "roc_auc": "{:.3f}",
        "precision_at_k": "{:.2%}",
        "recall_at_k": "{:.2%}",
    }
)

,model,cohort,sample_count,prevalence,pr_auc,roc_auc,top_k,precision_at_k,recall_at_k
0,base_rate,test_all,9293,5.62%,0.056,0.500,465,4.95%,4.41%
1,base_rate,test_seen,7085,4.12%,0.041,0.500,355,5.63%,6.85%
2,base_rate,test_cold_start,2208,10.42%,0.104,0.500,111,2.70%,1.30%
3,logistic_instrument,test_all,9293,5.62%,0.074,0.402,465,9.46%,8.43%
4,logistic_instrument,test_seen,7085,4.12%,0.096,0.566,355,11.83%,14.38%
5,logistic_instrument,test_cold_start,2208,10.42%,0.068,0.216,111,2.70%,1.30%
6,lightgbm_instrument_company,test_all,9293,5.62%,0.465,0.913,465,49.03%,43.68%
7,lightgbm_instrument_company,test_seen,7085,4.12%,0.432,0.900,355,34.65%,42.12%
8,lightgbm_instrument_company,test_cold_start,2208,10.42%,0.387,0.904,111,28.83%,13.91%


## 7. Reading the result honestly

The strong baseline is genuinely strong:

- **Overall:** LightGBM reaches PR-AUC 0.465 against 5.62% prevalence and ROC AUC 0.913. Its highest-risk 5% contains 49.03% impairments and retrieves 43.68% of all test impairments.
- **Seen companies:** PR-AUC 0.432 against 4.12% prevalence; precision@5% is 34.65%.
- **Cold start:** PR-AUC 0.387 and ROC AUC 0.904 are substantial, but prevalence is already 10.42%. Precision@5% is 28.83% and recall@5% only 13.91%. The high base rate makes this subgroup both risky and harder to triage deeply.
- **Logistic regression:** weak overall and actively poor under cold-start shift (PR-AUC 0.068 below 10.42% prevalence; ROC AUC 0.216). A linear invoice-only score does not transfer reliably to the new-company cohort.

The future GNN bar is therefore **0.465 overall PR-AUC**, with 0.432 seen and 0.387 cold-start reported alongside it. Beating only logistic regression would not justify a graph model.

## 8. What did LightGBM use?

Gain importance sums the loss reduction attributed to splits on each feature. It is useful for inspection but is **not causal** and can divide credit unpredictably among correlated variables.

In [3]:
gain_table = pd.DataFrame(
    run.lightgbm_feature_gains, columns=["feature", "gain"]
).head(12)
gain_table.assign(share=gain_table["gain"] / sum(
    gain for _, gain in run.lightgbm_feature_gains
)).style.format({"gain": "{:,.0f}", "share": "{:.1%}"})

,feature,gain,share
0,seller_endpoint__seller_history_log_count,"137,622",18.6%
1,buyer_endpoint__buyer_history_log_count,"125,922",17.0%
2,buyer_endpoint__buyer_history_mean_log_invoice_amount,"87,054",11.7%
3,log_invoice_amount,"64,151",8.7%
4,buyer_endpoint__buyer_history_mean_input_lag_days,"62,873",8.5%
5,buyer_endpoint__buyer_history_mean_payment_term_days,"48,586",6.6%
6,seller_endpoint__seller_history_mean_purchase_to_invoice_ratio,"41,298",5.6%
7,seller_endpoint__seller_history_mean_log_invoice_amount,"40,632",5.5%
8,buyer_endpoint__buyer_history_mean_purchase_to_invoice_ratio,"38,411",5.2%
9,seller_endpoint__seller_history_mean_input_lag_days,"28,720",3.9%


Company-history counts dominate the gain ranking, followed by buyer-history amount and timing summaries and the instrument's own amount. This supports the premise that relational context matters, but it does not yet prove message passing adds value: LightGBM obtained that context from fixed endpoint aggregates. The GNN must improve on this without receiving future outcomes or hand-engineered bond-graph features.

## 9. Verify the committed results artifact

The repository keeps a compact CSV result log so every headline number traces to a dated run and notebook. The assertion below checks that the committed metrics match this execution within the saved nine-decimal precision.

In [4]:
committed = pd.read_csv(repo_root / "results/baseline_metrics.csv")
metric_columns = [
    "prevalence",
    "pr_auc",
    "roc_auc",
    "precision_at_k",
    "recall_at_k",
]
current = report.sort_values(["model", "cohort"]).reset_index(drop=True)
saved = committed.sort_values(["model", "cohort"]).reset_index(drop=True)
np.testing.assert_allclose(
    current[metric_columns], saved[metric_columns], rtol=0, atol=5e-9
)
"Committed result log matches this run."

'Committed result log matches this run.'

## Takeaways

- Baselines form a ladder: no-skill floor, interpretable linear reference, then a serious nonlinear competitor.
- Validation must also respect time; test labels never choose LightGBM's tree count.
- LightGBM receives the same cutoff-safe raw and endpoint-history information available to the GNN, making the comparison meaningful.
- The overall LightGBM bar is PR-AUC 0.465, not the much weaker logistic result.
- Cold-start cases have much higher prevalence and require their own metrics; high ROC AUC alone does not tell the operational story.
- Feature importance suggests company context is useful, but only a GNN comparison can test whether learned graph aggregation improves on fixed endpoint summaries.